In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # project root, so `src` imports work
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
from src import config

# 03 — Visualization Pack (Deliverable C)

Figures `fig01`–`fig09` (data and analysis). `fig10`–`fig12` show model results, so they are drawn
at the end of `04_modeling_and_evaluation.ipynb`, which also completes `figures/figure_captions.md`.
Style: one colorblind-safe categorical palette (validated order), sequential single-hue maps for
magnitudes, bars from zero, 150 dpi.

In [2]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from src import analysis as A, cleaning, features, plotstyle as S
S.apply()
config.FIGURES.mkdir(exist_ok=True)
m = A.load_master()
events = A.load_events()
weather = A.load_weather()
obs = m[m.row_status == "observed"].copy()
o = A.expected_demand(m)
o_any = A.expected_demand(m, exclude_rain=False)
raw_train = pd.read_csv(config.RAW / "ride_demand_train.csv")
raw_weather = pd.read_csv(config.RAW / "weather_hourly.csv")
raw_events = pd.read_csv(config.RAW / "events_calendar.csv")
CAP = {}

## fig01 — Gaps and missingness

In [3]:
w_t, _ = cleaning.parse_weather_raw_clock(raw_weather["timestamp"])
day_med = raw_weather.groupby(w_t.dt.floor("D"))["temp_c"].transform("median")
tz = raw_train.assign(zc=raw_train.zone.map(cleaning.canonical_zone), t=cleaning.parse_trip_hours(raw_train.pickup_hour)[0])
ev_t = cleaning.parse_event_times(raw_events["start_datetime"])[0]; ev_e = cleaning.parse_event_times(raw_events["end_datetime"])[0]
issues = pd.DataFrame([
    ("Trips", "trips blank", raw_train.trips.isna().sum()),
    ("Trips", "trips = -1", (raw_train.trips == -1).sum()),
    ("Trips", "trips x8 spike", ((raw_train.trips / raw_train.active_drivers) >= cleaning.SPIKE_RATIO).sum()),
    ("Trips", "avg_wait_min = -1", (raw_train.avg_wait_min == -1).sum()),
    ("Trips", "avg_fare_birr blank", raw_train.avg_fare_birr.isna().sum()),
    ("Trips", "duplicate zone-hour", tz.duplicated(["zc", "t"], keep=False).sum()),
    ("Weather", "temp_c blank", raw_weather.temp_c.isna().sum()),
    ("Weather", "temp_c in degF", (day_med > cleaning.FAHRENHEIT_DAY_MEDIAN).sum()),
    ("Weather", "humidity_pct blank", raw_weather.humidity_pct.isna().sum()),
    ("Weather", "rain_mm = -9999", (raw_weather.rain_mm == -9999).sum()),
    ("Weather", "duplicate hour", w_t.duplicated(keep=False).sum()),
    ("Weather", "missing hour", len(pd.date_range(w_t.min(), w_t.max(), freq="h").difference(w_t))),
    ("Events", "end blank", raw_events.end_datetime.isna().sum()),
    ("Events", "end before start", (ev_e < ev_t).sum()),
    ("Events", "attendance blank", raw_events.expected_attendance.isna().sum()),
    ("Events", "duplicate event", raw_events.drop(columns="event_id").duplicated().sum()),
], columns=["table", "issue", "rows"])
issues["pct"] = issues.apply(lambda r: 100 * r.rows / {"Trips": len(raw_train), "Weather": len(raw_weather), "Events": len(raw_events)}[r.table], axis=1)

miss = m.assign(missing=(m.row_status != "observed").astype(int), day=m.pickup_hour.dt.floor("D"))
grid = miss.pivot_table(index="zone", columns="day", values="missing", aggfunc="sum")

fig = plt.figure(figsize=(15, 10))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.05], hspace=0.45, wspace=0.55)
for i, t in enumerate(["Trips", "Weather", "Events"]):
    ax = fig.add_subplot(gs[0, i])
    d = issues[issues.table == t].sort_values("pct")
    ax.barh(d.issue, d.pct, color=S.SERIES[i], height=0.6)
    for y, (p, n) in enumerate(zip(d.pct, d.rows)):
        ax.text(p, y, f"  {n:,}", va="center", fontsize=9.5, color=S.INK_2)
    ax.set_xlabel("% of raw rows"); ax.set_title(f"{t} file ({ {'Trips': len(raw_train), 'Weather': len(raw_weather), 'Events': len(raw_events)}[t]:,} rows)", fontsize=12)
    ax.set_xlim(0, d.pct.max() * 1.35); ax.grid(axis="y", visible=False)
ax = fig.add_subplot(gs[1, :])
im = ax.imshow(grid.values, aspect="auto", cmap=S.SEQ_CMAP, vmin=0, vmax=24, interpolation="nearest")
ax.set_yticks(range(len(grid.index))); ax.set_yticklabels(grid.index)
months = pd.date_range("2025-01-01", "2025-10-01", freq="MS")
ax.set_xticks([grid.columns.get_loc(d) for d in months]); ax.set_xticklabels([d.strftime("%b") for d in months])
ax.grid(False)
cb = fig.colorbar(im, ax=ax, pad=0.01); cb.set_label("missing hours per day")
o_day = grid.columns.get_loc(pd.Timestamp("2025-05-12"))
ax.annotate("platform outage\n12-13 May (42 h)", xy=(o_day + 1, 0), xytext=(o_day + 18, -2.3), fontsize=10,
            arrowprops=dict(arrowstyle="-", color=S.INK_2), annotation_clip=False)
ax.text(3, 1, "Ayat not launched until 15 Mar", va="center", fontsize=10, color="white", fontweight="bold")
ax.set_title("Missing zone-hours per day (no usable trips value), 1 Jan - 31 Oct 2025", fontsize=12)
ax.set_xlabel("date (2025)")
fig.suptitle("fig01  Gaps and invalid values in the three raw tables", fontsize=15, fontweight="bold", y=0.98)
S.save(fig, "fig01_gaps_and_missingness.png")
CAP["fig01_gaps_and_missingness.png"] = (
    f"Invalid or missing values per column in the three raw files (top; bar = % of rows, label = row count) and missing zone-hours per day "
    f"(bottom). The dark vertical stripe is the 42-hour platform outage on 12-13 May and the dark block is Ayat before its 15 March launch; "
    f"the remaining ~{int((m.row_status=='random_gap').sum()/12)} gaps per zone are scattered, so each gets its own treatment rather than a blanket fill.")
issues

saved fig01_gaps_and_missingness.png (2250x1500 px at 150 dpi)


,table,issue,rows,pct
0,Trips,trips blank,845,0.988767
1,Trips,trips = -1,681,0.796864
2,Trips,trips x8 spike,125,0.146267
3,Trips,avg_wait_min = -1,1707,1.997426
4,Trips,avg_fare_birr blank,1277,1.494266
5,Trips,duplicate zone-hour,1692,1.979874
6,Weather,temp_c blank,42,0.557177
7,Weather,temp_c in degF,354,4.696206
8,Weather,humidity_pct blank,153,2.029716
9,Weather,rain_mm = -9999,76,1.008225


## fig02 — Before vs after cleaning

In [4]:
clean_temp = weather.loc[weather.pickup_hour < config.TEST_START, "temp_c"]
raw_temp = raw_weather.loc[raw_weather.data_type == "observed", "temp_c"].dropna()
raw_ratio = (raw_train.trips / raw_train.active_drivers).replace([np.inf, -np.inf], np.nan).dropna()
raw_ratio = raw_ratio[raw_train.loc[raw_ratio.index, "trips"] >= 0]
clean_ratio = (obs.trips / obs.active_drivers)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
ax = axes[0]
bins = np.arange(0, 80, 1)
ax.hist(raw_temp, bins=bins, color=S.SERIES[1], alpha=0.55, label=f"raw ({(raw_temp > 40).sum()} readings > 40, in degF)")
ax.hist(clean_temp, bins=bins, histtype="step", color=S.SERIES[0], lw=2.2, label="cleaned (degF block converted)")
ax.set_xlabel("air temperature (degC)"); ax.set_ylabel("hours"); ax.set_title("Temperature: a 15-day degF block in July")
ax.legend(loc="upper right")
ax = axes[1]
bins = np.arange(0, 15, 0.1)
ax.hist(raw_ratio, bins=bins, color=S.SERIES[1], alpha=0.55, label=f"raw ({(raw_ratio >= 4).sum()} rows >= 4, x8 export error)")
ax.hist(clean_ratio, bins=bins, histtype="step", color=S.SERIES[0], lw=2.2, label="cleaned (spikes divided by 8)")
ax.set_yscale("log"); ax.set_xlabel("trips per active driver in the zone-hour"); ax.set_ylabel("zone-hours (log scale)")
ax.set_title("Trips: inflated counts at 10-14 trips per driver"); ax.legend(loc="upper right")
fig.suptitle("fig02  Before and after cleaning", fontsize=15, fontweight="bold")
fig.tight_layout()
S.save(fig, "fig02_before_after_cleaning.png")
CAP["fig02_before_after_cleaning.png"] = (
    f"Raw vs cleaned distributions. Left: {(raw_temp > 40).sum()} July temperatures were stored in degF (54-77), creating a second hump that "
    "disappears after conversion. Right: on a log scale, a separate cluster of zone-hours at 10-14 trips per driver (normal is ~1.3) is the "
    f"x8 export error on {(raw_ratio >= 4).sum()} rows; after dividing by 8 it merges into the main distribution. Without these fixes the model "
    "would learn fake heat waves and fake demand spikes.")

saved fig02_before_after_cleaning.png (2250x840 px at 150 dpi)


## fig03 — Demand trend with holidays

In [5]:
older = obs[obs.zone != "Ayat"]
daily = older.groupby(older.pickup_hour.dt.floor("D"))["trips"].agg(["sum", "size"])
daily = daily[daily["size"] >= 11 * 24 * 0.9]     # drop days hit by the outage
roll = daily["sum"].rolling(7, center=True).mean()
x = (daily.index - daily.index[0]).days.values
b1, b0 = np.polyfit(x, daily["sum"].values, 1)
hol = events[(events.event_type == "public_holiday") & (events.status == "confirmed") & (events.start <= config.TRAIN_END)]
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(daily.index, daily["sum"], color=S.SERIES[0], alpha=0.35, lw=1, label="daily trips (11 zones open all year)")
ax.plot(roll.index, roll, color=S.SERIES[0], lw=2.2, label="7-day centred mean")
ax.plot(daily.index, b0 + b1 * x, color=S.SERIES[1], lw=2, ls="--", label=f"linear trend (daily trips rise by {b1*7:,.0f} each week)")
for i, r in enumerate(hol.sort_values("start").itertuples()):
    d = r.start.floor("D")
    ax.axvline(d, color=S.SERIES[6], lw=1, ls=":")
    ax.text(d + pd.Timedelta(hours=10), daily["sum"].max() * 1.03, r.event_name.split(" (")[0], rotation=90,
            fontsize=8.5, ha="center", va="bottom", color=S.SERIES[6])
ax.plot([], [], color=S.SERIES[6], ls=":", label="public holiday")
ax.set_ylim(0, daily["sum"].max() * 1.42)
ax.set_ylabel("trips per day"); ax.set_xlabel("date (2025)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.legend(loc="lower right", ncol=2)
ax.set_title("fig03  City-wide daily trips, January-October 2025 (outage days removed)")
S.save(fig, "fig03_demand_trend_with_holidays.png")
growth = (daily["sum"].iloc[-28:].mean() / daily["sum"].iloc[:28].mean() - 1) * 100
CAP["fig03_demand_trend_with_holidays.png"] = (
    f"Daily trips in the 11 zones open all year, with a 7-day mean and a linear trend. Demand grows steadily (the trend line adds about "
    f"{b1:,.0f} trips/day every day, +{growth:.0f}% from the first to the last 4 weeks), and most holidays show up as one-day dips. "
    "So what: November will be busier than any month in the training data, and the model needs a recent-level feature to keep up.")

saved fig03_demand_trend_with_holidays.png (2250x900 px at 150 dpi)


## fig04 — Hour x weekday heatmaps

In [6]:
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
panels = [("City-wide (mean of 12 zones)", obs), ("Kazanchis (business district)", obs[obs.zone == "Kazanchis"]),
          ("Bole (nightlife / airport)", obs[obs.zone == "Bole"])]
fig, axes = plt.subplots(1, 3, figsize=(17, 6.2), sharey=True)
for ax, (title, d) in zip(axes, panels):
    h = d.pivot_table(index="hour", columns="dow", values="trips", aggfunc="mean")
    im = ax.imshow(h.values, aspect="auto", cmap=S.SEQ_CMAP, vmin=0, interpolation="nearest")
    ax.set_xticks(range(7)); ax.set_xticklabels(days); ax.set_yticks(range(0, 24, 3))
    ax.set_yticklabels([f"{x:02d}:00" for x in range(0, 24, 3)]); ax.grid(False)
    ax.set_title(title, fontsize=12.5); ax.set_xlabel("day of week")
    cb = fig.colorbar(im, ax=ax, fraction=0.05, pad=0.02); cb.set_label("mean trips per zone-hour")
axes[0].set_ylabel("hour of day (EAT)")
fig.suptitle("fig04  Mean trips by hour of day x day of week", fontsize=15, fontweight="bold")
fig.tight_layout()
S.save(fig, "fig04_hour_by_weekday_heatmap.png")
CAP["fig04_hour_by_weekday_heatmap.png"] = (
    "Mean trips per zone-hour by hour and weekday, city-wide and for two contrasting zones. The city shows weekday commute peaks at "
    "08:00 and 18:00; Kazanchis is a pure weekday-office zone that goes quiet at weekends, while Bole is busiest on Friday and Saturday "
    "nights and early mornings. So what: hour x weekday x zone interactions carry most of the predictable signal.")

saved fig04_hour_by_weekday_heatmap.png (2550x930 px at 150 dpi)


## fig05 — Zone-type profiles

In [7]:
types = features.ZONE_TYPE_ORDER
fig, axes = plt.subplots(1, 5, figsize=(19, 4.9), sharey=False)
for ax, t in zip(axes, types):
    d = obs[obs.zone_type == t]
    for we, color, lab in [(0, S.SERIES[0], "weekday"), (1, S.SERIES[1], "weekend")]:
        p = d[d.is_weekend == we].groupby("hour")["trips"].mean()
        ax.plot(p.index, p.values, color=color, label=lab)
    ax.set_title(S.ZONE_TYPE_LABELS[t] + "\n(" + ", ".join(sorted(d.zone.unique())) + ")", fontsize=11.5)
    ax.set_xticks([0, 6, 12, 18, 23]); ax.set_xlabel("hour of day (EAT)"); ax.set_ylim(0, None)
axes[0].set_ylabel("mean trips per zone-hour")
axes[0].legend(loc="upper left")
fig.suptitle("fig05  Weekday vs weekend hourly profile by zone type", fontsize=15, fontweight="bold")
fig.tight_layout()
S.save(fig, "fig05_zone_profiles.png")
CAP["fig05_zone_profiles.png"] = (
    "Mean hourly trips on weekdays and weekends for each zone type (each panel has its own y-axis starting at zero). Business districts "
    "lose their commute peaks at the weekend, residential zones barely change, the market is a midday plateau, and nightlife/airport (Bole) "
    "is higher at the weekend at almost every hour. So what: one city-wide daily curve would be wrong for every zone type.")

saved fig05_zone_profiles.png (2850x735 px at 150 dpi)


## fig06 — Weather clock check

In [8]:
w_raw = raw_weather.assign(t=w_t)
plaus = w_raw.temp_c.between(-5, 35)
raw_curve = w_raw[plaus].groupby(w_raw.t.dt.hour)["temp_c"].mean()
eat_curve = w_raw[plaus].groupby((w_raw.t + pd.Timedelta(hours=3)).dt.hour)["temp_c"].mean()
# rain-demand correlation vs shift
w = weather.set_index("pickup_hour")
zh = o_any[(o_any.zone_type != "market") & o_any.hour.between(6, 22)]
corr = {}
for k in range(0, 7):
    r = w["rain_mm"].copy(); r.index = r.index - pd.Timedelta(hours=3) + pd.Timedelta(hours=k)
    v = r.reindex(zh.pickup_hour).values; ok = ~np.isnan(v)
    corr[k] = np.corrcoef(v[ok], zh.ratio.values[ok])[0, 1]
corr = pd.Series(corr)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
ax = axes[0]
ax.plot(raw_curve.index, raw_curve.values, color=S.SERIES[1], marker="o", ms=5, label=f"raw clock (as written): peak {raw_curve.idxmax()}:00")
ax.plot(eat_curve.index, eat_curve.values, color=S.SERIES[0], marker="o", ms=5, label=f"corrected, UTC+3 = EAT: peak {eat_curve.idxmax()}:00")
ax.set_xticks(range(0, 24, 3)); ax.set_xlabel("hour of day"); ax.set_ylabel("mean air temperature (degC)")
ax.set_title("Daily temperature curve"); ax.legend(loc="lower center")
ax = axes[1]
colors = [S.SERIES[0] if k == 3 else S.NEUTRAL for k in corr.index]
ax.bar(corr.index, corr.values, color=colors, width=0.6)
for k, v in corr.items():
    ax.text(k, v + 0.005, f"{v:.2f}", ha="center", fontsize=10)
ax.set_xticks(corr.index); ax.set_xticklabels([f"+{k} h" + ("\n(UTC->EAT)" if k == 3 else "") for k in corr.index])
ax.set_xlabel("hours added to the raw weather timestamp"); ax.set_ylabel("correlation of rain with demand ratio")
ax.set_ylim(0, corr.max() * 1.18); ax.set_title("Rain-demand correlation vs clock shift")
fig.suptitle("fig06  The weather table is on UTC: shifting it +3 h aligns it with the trips (EAT)", fontsize=15, fontweight="bold")
fig.tight_layout()
S.save(fig, "fig06_weather_timezone_check.png")
CAP["fig06_weather_timezone_check.png"] = (
    f"Left: on the raw clock temperature peaks at {raw_curve.idxmax()}:00, too early for Addis; shifted to EAT it peaks at {eat_curve.idxmax()}:00. "
    f"Right: rain only lines up with demand when the weather clock is shifted by exactly +3 h (correlation {corr[3]:.2f} vs {corr.drop(3).max():.2f} or less "
    "at any other shift). So what: joining on the raw timestamps would have thrown away almost all of the rain signal.")

saved fig06_weather_timezone_check.png (2250x840 px at 150 dpi)


## fig07 — Rain effect

In [9]:
oa = o_any.copy()
oa["month"] = oa.pickup_hour.dt.month
clean_slot = (oa[A.EVENT_WINDOW_COLS].sum(axis=1) == 0) & (oa.is_public_holiday == 0)
dry_ref = oa[clean_slot & (oa.rain_mm == 0)].groupby(["zone", "month", "dow", "hour"])["trips"].mean().rename("dry_ref")
rr = oa[clean_slot].join(dry_ref, on=["zone", "month", "dow", "hour"]).dropna(subset=["dry_ref"])
rr["r"] = rr.trips / rr.dry_ref
dose = rr.groupby(["zone_type", "rain_class"])["r"].agg(["mean", "count", "std"]).reset_index()
dose["se"] = dose["std"] / np.sqrt(dose["count"])
fig, ax = plt.subplots(figsize=(13, 6))
classes = [1, 2, 3]; width = 0.16
for i, t in enumerate(features.ZONE_TYPE_ORDER):
    d = dose[(dose.zone_type == t) & dose.rain_class.isin(classes)].set_index("rain_class").reindex(classes)
    xs = np.arange(len(classes)) + (i - 2) * (width + 0.01)
    ax.bar(xs, d["mean"], width=width, color=S.ZONE_TYPE_COLORS[t], label=S.ZONE_TYPE_LABELS[t],
           yerr=1.96 * d["se"], error_kw=dict(ecolor=S.INK_2, lw=1, capsize=2))
ax.axhline(1, color=S.INK, lw=1.2, ls="--"); ax.text(2.55, 1.02, "no effect", fontsize=10, ha="right")
ax.set_xticks(range(3)); ax.set_xticklabels(["light (0-2.5 mm/h)", "moderate (2.5-7.5)", "heavy (> 7.5)"])
ax.set_xlabel("rain class in the hour"); ax.set_ylabel("trips / matched dry-hour trips")
ax.set_ylim(0, None); ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.13))
ax.set_title("fig07  Demand in rain vs the same zone, weekday and hour when dry (95% CI)")
S.save(fig, "fig07_rain_effect.png")
mk = dose[(dose.zone_type == "market") & (dose.rain_class == 3)]["mean"].iloc[0]
oth = dose[(dose.zone_type != "market") & (dose.rain_class == 3)]["mean"].mean()
CAP["fig07_rain_effect.png"] = (
    f"Demand ratio in light, moderate and heavy rain vs matched dry hours (same zone, weekday, hour, month), by zone type. Rain lifts demand "
    f"everywhere except the open-air market, and the lift grows with intensity (heavy rain: {oth:.2f}x outside the market, {mk:.2f}x in Merkato). "
    "So what: rain is worth joining, but only with a zone-type interaction.")

saved fig07_rain_effect.png (1950x900 px at 150 dpi)


## fig08 — Event study

In [10]:
offs = range(-6, 9)
fig, ax = plt.subplots(figsize=(13, 6))
summary = {}
for i, t in enumerate(["football_match", "concert", "conference"]):
    es = A.event_study(o, events, t, offsets=offs)
    ax.plot(es.index, es["mean"], color=S.SERIES[i], marker="o", ms=5,
            label=f"{t.replace('_', ' ')} (n = {es.attrs['n_events']} events)")
    ax.fill_between(es.index, es["mean"] - 1.96 * es["se"], es["mean"] + 1.96 * es["se"], color=S.SERIES[i], alpha=0.15, lw=0)
    summary[t] = es["mean"]
ax.axhline(1, color=S.INK, lw=1.2, ls="--", label="no-event baseline (= 1)")
ax.axvline(0, color=S.INK_2, lw=1, ls=":")
ax.text(0.1, ax.get_ylim()[1] * 0.97, "event starts", fontsize=10, color=S.INK_2, va="top")
ax.set_xticks(list(offs)); ax.set_xlabel("hours from event start"); ax.set_ylabel("trips / expected trips without event")
ax.set_ylim(0, None); ax.legend(loc="upper left")
ax.set_title("fig08  Demand in the event's zone around the start time (95% band)")
S.save(fig, "fig08_event_study.png")
s = pd.DataFrame(summary)
CAP["fig08_event_study.png"] = (
    f"Average demand ratio in the event zone from 6 h before to 8 h after the start (1 = normal hour). Football peaks twice: arrivals "
    f"({s['football_match'].loc[-2:-1].max():.1f}x, 1-2 h before kick-off) and departures ({s['football_match'].loc[2:3].max():.1f}x right after the 2-hour match); "
    f"concerts peak as the crowd leaves around {s['concert'].idxmax():+d} h ({s['concert'].max():.1f}x) and conferences add a smaller morning lift. "
    "So what: event features must be split into before/during/after phases, not a single in-event flag.")

saved fig08_event_study.png (1950x900 px at 150 dpi)


## fig09 — Holiday effects

In [11]:
daily_mean = obs.groupby(obs.pickup_hour.dt.floor("D"))["trips"].mean()
hol_days = set(hol.start.dt.floor("D"))
rows = []
for r in hol.itertuples():
    d = r.start.floor("D")
    refs = [d + pd.Timedelta(weeks=k) for k in (-2, -1, 1, 2)]
    refs = [x for x in refs if x in daily_mean.index and x not in hol_days]
    rows.append((r.event_name.split(" (")[0], d, daily_mean[d] / daily_mean[refs].mean()))
hi = pd.DataFrame(rows, columns=["holiday", "date", "index"]).sort_values("index")
fig, ax = plt.subplots(figsize=(12, 7))
cols = [S.SERIES[1] if v < 1 else S.SERIES[0] for v in hi["index"]]
ax.barh([f"{h} ({d:%d %b}, {d:%a})" for h, d in zip(hi.holiday, hi.date)], hi["index"], color=cols, height=0.6)
for y, v in enumerate(hi["index"]):
    ax.text(max(v, 1) + 0.012, y, f"{v:.2f}", va="center", fontsize=10)
ax.axvline(1, color=S.INK, lw=1.2, ls="--")
ax.set_xlim(0, 1.25); ax.set_xlabel("city-wide trips per zone-hour vs the same weekday 1-2 weeks before/after (1 = normal)")
ax.grid(axis="y", visible=False)
ax.plot([], [], color=S.SERIES[1], lw=8, label="below normal"); ax.plot([], [], color=S.SERIES[0], lw=8, label="at or above normal")
ax.legend(loc="lower right")
ax.set_title("fig09  Demand index on each public holiday, sorted")
S.save(fig, "fig09_holiday_effects.png")
CAP["fig09_holiday_effects.png"] = (
    f"City-wide trips on each holiday relative to the same weekday in nearby weeks. Most weekday holidays cut demand by 10-26% "
    f"({hi.iloc[0].holiday} is the deepest at {hi.iloc[0]['index']:.2f}), while the three Sunday holidays barely move it (Adwa Victory Day even rises to {hi['index'].max():.2f}). "
    "So what: a holiday flag helps, but its effect depends on the weekday and (B3.1) the zone.")

saved fig09_holiday_effects.png (1800x1050 px at 150 dpi)


## Captions

In [12]:
from pathlib import Path
lines = ["# Figure captions (team " + config.TEAM_NAME + ")", "",
         "One caption per figure: what it shows and the takeaway. Figures 01-09 are drawn by "
         "`notebooks/03_visualizations.ipynb`, figures 10-12 by `notebooks/04_modeling_and_evaluation.ipynb`.", ""]
for name, text in CAP.items():
    lines += [f"## {name}", "", text, ""]
(config.FIGURES / "figure_captions.md").write_text("\n".join(lines), encoding="utf-8")
print(len(CAP), "captions written")

9 captions written
